<a href="https://colab.research.google.com/github/MouseLand/cellpose/blob/main/notebooks/run_Cellpose-SAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Run Cellpose-SAM

Adapted from Marius Pachitariu, Michael Rariden, Carsen Stringer and the notebook by Pradeep Rajasekhar, inspired by the [ZeroCostDL4Mic notebook series](https://github.com/HenriquesLab/ZeroCostDL4Mic/wiki)

[paper](https://www.biorxiv.org/content/10.1101/2025.04.28.651001v1) | [code](https://github.com/MouseLand/cellpose)

### Make sure you are in the correct environment


In [ ]:
# Check GPU and instantiate model - will download weights.
import numpy as np
from cellpose import models, core, io, plot
from pathlib import Path
from tqdm import trange
import matplotlib.pyplot as plt
import cv2 as cv 
import tifffile as tf
%matplotlib inline
from natsort import natsorted

io.logger_setup() # run this to get printing of progress

#Check if GPU access
if core.use_gpu()==False:
  raise ImportError("No GPU access, change your runtime")

model = models.CellposeModel(gpu=True)

Input directory with your images:
- Note - For best accuracy and runtime performance, resize images so cells are less than 100 pixels across

In [ ]:
#Inputs
dir = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/MRC-5_MAX_SUM_PROJ/20250328"
dir = Path(dir)
maskdir = dir / "masks"
maskdir.mkdir(exist_ok=True)

# *** change to your image extension ***
image_ext = ".tif"
nchannels = 4


In [ ]:
# set up the correct order for the image filenames - sort by location first, then channel
def file_sort_key(filename):
  parts = filename.split("-")
  channel = parts[0][-1:] # get the last character of the first part
  location = parts[1]
  return (location,channel)

def plate_location(filename):
  parts = filename.split("-")
  pre_location = parts[1]
  location = pre_location.split(".")[0] # get the first part of the second part
  return location
  
#list all files
def sort_files(dir, image_ext):
  if not dir.exists():
    raise FileNotFoundError("directory does not exist")
  files = sorted([f for f in dir.glob("*"+image_ext) if "_masks" not in f.name and "_flows" not in f.name and "SUM" not in f.name],
                           key=lambda x: file_sort_key(x.name))
 # sort by number in filename
  if(len(files)==0):
    raise FileNotFoundError("no image files found, did you specify the correct folder and extension?")
  else:
    return files
  
def print_files(files):
  for f in files:
    print(f.name)

def group_files_by_channel(files, nchannels=4):
  grouped = []
  for i in range(0,len(files),nchannels):
    grouped.append(files[i:i+nchannels])
  return grouped

def print_grouped_files(grouped):
  for i in range(len(grouped)):
    print(f"\n Group {i+1} of {len(grouped)}")
    for j in range(len(grouped[i])):
      item = grouped[i][j]
      print(" "+ item.name)
    
files = sort_files(dir, image_ext)
grouped_files = group_files_by_channel(files, nchannels)

print_files(files)
print_grouped_files(grouped_files)
plate_location(files[0].name)

In [ ]:
# load the images from a group of image filenames
def load_image_set(file_group,nchannels=4):
  # load the images from the channels - skip ch3 at position 2 as DAPI is always the last channel
  lastindex = nchannels - 1
  ch1,ch2,ch3 = io.imread(file_group[0]), io.imread(file_group[1]), io.imread(file_group[lastindex])
  image_set = [ch1,ch2*2,ch3]
  return image_set

def get_image_set_name(file_group, index=0):
  # get the name of the image set from the filename
  set_name = plate_location(file_group[index].name)
  return set_name

def save_masks(set_name, masks, image_ext=".tif", mask_type="cell", dir=maskdir):
    # save the masks to a file
    masks_ext = ".png" if image_ext == ".png" else ".tif"
    io.imsave(dir / (set_name + "_" + mask_type + "_masks" + masks_ext), masks)
    


## Run Cellpose-SAM on one image in folder

Here are some of the parameters you can change:

* ***flow_threshold*** is  the  maximum  allowed  error  of  the  flows  for  each  mask.   The  default  is 0.4.
    *  **Increase** this threshold if cellpose is not returning as many masks as you’d expect (or turn off completely with 0.0)
    *   **Decrease** this threshold if cellpose is returning too many ill-shaped masks.

* ***cellprob_threshold*** determines proability that a detected object is a cell.   The  default  is 0.0.
    *   **Decrease** this threshold if cellpose is not returning as many masks as you’d expect or if masks are too small
    *   **Increase** this threshold if cellpose is returning too many masks esp from dull/dim areas.

* ***tile_norm_blocksize*** determines the size of blocks used for normalizing the image. The default is 0, which means the entire image is normalized together.
  You may want to change this to 100-200 pixels if you have very inhomogeneous brightness across your image.



In [ ]:
def img_preprocessing(channels):
    from skimage import io, exposure, filters, morphology
    #ch1,ch2,ch3 = io.imread(files[0]), io.imread(files[1]), io.imread(files[3])
    #channels = [ch1, ch2, ch3]
    for i in range(len(channels)):
        footprint = morphology.disk(2)
        channel = channels[i]
        channel = img_01_normalization(channel)
        channel = exposure.equalize_adapthist(channel, kernel_size=100, clip_limit=0.05)
        #channel = exposure.equalize_hist(channel)
        channel = filters.gaussian(channel, sigma=2)
        #channel = filters.median(channel, footprint=footprint)
        
        
        channels[i] = channel

    multi_channel_image = np.stack(channels, axis=-1)

    #rescaled_image = exposure.rescale_intensity(multi_channel_image, out_range=(0, 255))
    return multi_channel_image
    
    
def img_z_normalization(img):
    # Normalize each channel to z score
    norm_img = (img - np.mean(img)) / np.std(img)
    return norm_img

def img_01_normalization(img):
    # Normalize each channel to the range [0, 1]
    norm_img = (img - np.min(img)) / (np.max(img) - np.min(img))
    return norm_img

def img_rescaled(img, factor=0.5):
    from skimage import transform
    rescaled_img = transform.rescale(img, factor, anti_aliasing=False, channel_axis=-1)
    return rescaled_img  

image_set_index = 10
in_channels = load_image_set(grouped_files[image_set_index-1])
set_name = get_image_set_name(grouped_files[image_set_index-1])
print("Set name: ", set_name)

img1 = img_preprocessing(in_channels)
img2 = img_rescaled(img1, factor=0.25)
tf.imshow(img1)
tf.imshow(img2)

In [ ]:
img = img2
#img = io.imread(files[0])
def segment_cell(img, show=True):
    flow_threshold = 0.5
    cellprob_threshold = -1
    tile_norm_blocksize = 0
    diameter = 65

    masks, flows, styles = model.eval(img, batch_size=32, diameter=diameter, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold,
                                    normalize={"tile_norm_blocksize": tile_norm_blocksize})
    #plot if true
    if show:
        fig = plt.figure(figsize=(12,5))
        plot.show_segmentation(fig, img, masks, flows[0])
        plt.tight_layout()
        plt.show()
    return masks
    
def segment_nuclei(orig_img, show=True):
    from skimage import morphology, feature, filters
    img = orig_img[:,:,2] # get the DAPI channel
    
    # remove background
    dog = filters.difference_of_gaussians(img, low_sigma=2.5)
    seed = np.minimum(dog, img)  # ensure seed is not greater than the original image
    bg = morphology.reconstruction(seed, img, method='dilation')
    img = img - bg
    
    # remove speckle-shaped autofluor
    bg2 = morphology.white_tophat(img, morphology.disk(3))
    img = img - bg2
    img = morphology.closing(img, morphology.disk(2.5))
    img = filters.gaussian(img, sigma=1)
    
    flow_threshold = 0.4
    cellprob_threshold = 0
    tile_norm_blocksize = 0
    diameter = None

    masks, flows, styles = model.eval(img, batch_size=32, diameter=diameter, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold,
                                    normalize={"tile_norm_blocksize": tile_norm_blocksize})
    if show:
        fig = plt.figure(figsize=(12,5))
        plot.show_segmentation(fig, img, masks, flows[0])
        plt.tight_layout()
        plt.show()
    return masks
    

from skimage import feature, filters   
cell_masks = segment_cell(img)
nuc_masks = segment_nuclei(img)
#save_masks(set_name, cell_masks, image_ext=image_ext)
#save_masks(set_name, nuc_masks, image_ext=image_ext, mask_type="nuclei")
#blobs = filters.median(img[:,:,1])
#tf.imshow(blobs)

## Run Cellpose-SAM on folder of images

if you have many large images, you may want to run them as a loop over images



In [ ]:
# loop for all files in the group in the directory
for i in trange(len(grouped_files)):
    file_group = grouped_files[i]
    img_set = load_image_set(file_group)
    img_set_name = get_image_set_name(file_group)
    print("Set name: ", set_name)
    
    stacked_img = img_preprocessing(img_set)
    rescaled_img = img_rescaled(stacked_img, factor=0.25)
    
    cell_masks = segment_cell(rescaled_img, show=False)
    nuc_masks = segment_nuclei(rescaled_img, show=False) 
    
    save_masks(img_set_name, cell_masks, image_ext=image_ext)
    save_masks(img_set_name, nuc_masks, image_ext=image_ext, mask_type="nuclei") 